
#Setting Up the Models



##Install Libraries and Import Packages

In [ ]:
!pip install -r requirements_rg.txt

In [ ]:
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeRegressor
from interpret.glassbox import ExplainableBoostingRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import  mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

from interpret import show
import shap
import sage
import graphviz
from sklearn.tree import export_graphviz

from scipy.stats import spearmanr
from collections import Counter

import random

import warnings
warnings.filterwarnings('ignore')

# fetch dataset
parkinsons_telemonitoring = fetch_ucirepo(id=189)

# data (as pandas dataframes)
df = parkinsons_telemonitoring.data.original.copy()

state = 100
SEED = 100
np.random.seed(SEED)
random.seed(SEED)

##Dataset Analysis

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

##Handling Outliers with IQR Method

In [ ]:
for col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] > upper, upper, df[col])
    df[col] = np.where(df[col] < lower, lower, df[col])

##Splitting Data

In [ ]:
X = df.drop(['total_UPDRS', 'motor_UPDRS', 'subject#', 'age'], axis=1)
y = df['total_UPDRS']

#change column names for LightGBM
X.columns = ["".join (c if c.isalnum() else "_" for c in str(x)) for x in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

##Creating the Optimized Models and Then Fitting them

In [ ]:
dt = DecisionTreeRegressor(ccp_alpha=0.0, criterion='squared_error', max_depth=19,
                           max_features=None, max_leaf_nodes=None, min_impurity_decrease=0.0,
                           min_samples_leaf=14, min_samples_split=5, min_weight_fraction_leaf=0.0,
                           monotonic_cst=None, random_state=100, splitter='random')

ebm = ExplainableBoostingRegressor(callback=None, cat_smooth=10.0, cyclic_progress=False,
                                   early_stopping_rounds=100, early_stopping_tolerance=1e-05, exclude=None,
                                   feature_names=None, feature_types=None, gain_scale=5.0,
                                   greedy_ratio=10.0, inner_bags=0, interaction_smoothing_rounds=100,
                                   interactions=6, learning_rate=0.0027763309052116383, max_bins=484,
                                   max_delta_step=0.0, max_interaction_bins=263, max_leaves=28,
                                   max_rounds=50000, min_cat_samples=10, min_hessian=0.0,
                                   min_samples_leaf=15, missing='separate', monotone_constraints=None,
                                   n_jobs=-2, objective='rmse', outer_bags=14,
                                   random_state=100, reg_alpha=0.0, reg_lambda=0.0,
                                   smoothing_rounds=500, validation_size=0.15)

cat = CatBoostRegressor(iterations=816, learning_rate=0.1261192471208588, depth=10,
                        l2_leaf_reg=9.06507208082491, loss_function='RMSE', bootstrap_type='MVS',
                        random_state=100)

lgbm = LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=0.7507121131532456,
                     importance_type='split', learning_rate=0.042853382162136154, max_depth=18,
                     min_child_samples=8, min_child_weight=0.001, min_split_gain=0.0,
                     n_estimators=768, n_jobs=None, num_leaves=355,
                     objective=None, random_state=100, reg_alpha=2.592134383897951,
                     reg_lambda=6.6855134447038855, subsample=0.9198022636755252, subsample_for_bin=200000,
                     subsample_freq=0)

In [ ]:
dt.fit(X_train, y_train)
ebm.fit(X_train, y_train)
cat.fit(X_train, y_train)
lgbm.fit(X_train, y_train)

##Evaluating Each Model's Performance

In [ ]:
dt_pred = dt.predict(X_test)
print("For the Decision Tree Model:")
print("MSE is : " + str(mean_squared_error(y_test, dt_pred)))
print("R2 is : " + str(r2_score(y_test, dt_pred)))
print("MAE is : " + str(mean_absolute_error(y_test, dt_pred)))

In [ ]:
ebm_pred = ebm.predict(X_test)
print("For the Explainable Boosting Machine Model:")
print("MSE is : " + str(mean_squared_error(y_test, ebm_pred)))
print("R2 is : " + str(r2_score(y_test, ebm_pred)))
print("MAE is : " + str(mean_absolute_error(y_test, ebm_pred)))

In [ ]:
cat_pred = cat.predict(X_test)
print("For the CatBoost Model:")
print("MSE is : " + str(mean_squared_error(y_test, cat_pred)))
print("R2 is : " + str(r2_score(y_test, cat_pred)))
print("MAE is : " + str(mean_absolute_error(y_test, cat_pred)))

In [ ]:
lgbm_pred = lgbm.predict(X_test)
print("For the LightGBM Model:")
print("MSE is : " + str(mean_squared_error(y_test, lgbm_pred)))
print("R2 is : " + str(r2_score(y_test, lgbm_pred)))
print("MAE is : " + str(mean_absolute_error(y_test, lgbm_pred)))

#Global Explainability

##Decision Tree's Tree Structure

In [ ]:
dot_data = export_graphviz(dt, out_file=None,
                                feature_names=X.columns,
                                filled=True)

graph = graphviz.Source(dot_data, format="png")
graph

##EBM Global Explainability

In [ ]:
ebm_global = ebm.explain_global()
show(ebm_global)

##Setting Up SHAP explainers

In [ ]:
#turn data into DF for SHAP plots
X_test_df = pd.DataFrame(X_test, columns=X.columns)

dt_explainer = shap.TreeExplainer(dt)
dt_shap_values = dt_explainer.shap_values(X_test_df)

#wrapper function for EBM predict to handle feature names
def ebm_predict_wrapper(X):
    X_df = pd.DataFrame(X, columns=X_train.columns)
    return ebm.predict(X_df)

background = shap.sample(X_train, 100, random_state=SEED)
ebm_explainer = shap.KernelExplainer(ebm_predict_wrapper, background, seed=SEED)
ebm_shap_values = ebm_explainer.shap_values(X_test_df)

cat_explainer = shap.TreeExplainer(cat)
cat_shap_values = cat_explainer.shap_values(X_test_df)

lgbm_explainer = shap.TreeExplainer(lgbm)
lgbm_shap_values = lgbm_explainer.shap_values(X_test_df)

##Setting Up SAGE Explainers

In [ ]:
rng = np.random.RandomState(SEED)
background_idx = rng.choice(len(X_train), 512, replace=False)
X_sage_bg = X_train.values[background_idx]

dt_imputer = sage.MarginalImputer(dt, X_sage_bg)
dt_estimator = sage.PermutationEstimator(dt_imputer, 'mse', random_state=state)
dt_sage_values = dt_estimator(X_test.values, y_test.values)

ebm_imputer = sage.MarginalImputer(ebm, X_sage_bg)
ebm_estimator = sage.PermutationEstimator(ebm_imputer, 'mse', random_state=state)
ebm_sage_values = ebm_estimator(X_test.values, y_test.values)

cat_imputer = sage.MarginalImputer(cat, X_sage_bg)
cat_estimator = sage.PermutationEstimator(cat_imputer, 'mse', random_state=state)
cat_sage_values = cat_estimator(X_test.values, y_test.values)

lgbm_imputer = sage.MarginalImputer(lgbm, X_sage_bg)
lgbm_estimator = sage.PermutationEstimator(lgbm_imputer, 'mse', random_state=state)
lgbm_sage_values = lgbm_estimator(X_test.values, y_test.values)

##SHAP Summary Plots For Each Model

In [ ]:
shap.summary_plot(dt_shap_values, X_test_df)

In [ ]:
shap.summary_plot(ebm_shap_values, X_test_df)

In [ ]:
shap.summary_plot(cat_shap_values, X_test_df)

In [ ]:
shap.summary_plot(lgbm_shap_values, X_test_df)

##SAGE Summary Plots For Each Model

In [ ]:
dt_sage_values.plot(X.columns, title="Feature Importance (Decision Tree Model)")

In [ ]:
ebm_sage_values.plot(X.columns, title="Feature Importance (Explainable Boosting Machine Model)")

In [ ]:
cat_sage_values.plot(X.columns, title="Feature Importance (CatBoost Model)")

In [ ]:
lgbm_sage_values.plot(X.columns, title="Feature Importance (LightGBM Model)")

#Local Explainability

##Selecting Instance

In [ ]:
index = 0

##Decision Path

In [ ]:
X_instance = X_test.iloc[[index]]

node_indicator = dt.decision_path(X_instance)
leaf_id = dt.apply(X_instance)

print("Decision path for instance " + str(index) + ":")
for node_id in node_indicator.indices:
    if dt.tree_.children_left[node_id] != dt.tree_.children_right[node_id]:
        feature = X_test.columns[dt.tree_.feature[node_id]]
        threshold = dt.tree_.threshold[node_id]
        if X_instance.iloc[0, dt.tree_.feature[node_id]] <= threshold:
            threshold_sign = "<="
        else:
            threshold_sign = ">"
        print(str(feature) + " = " + str(X_instance.iloc[0, dt.tree_.feature[node_id]]) + " "
              + str(threshold_sign) + " "  + str(threshold))

pred_value = dt.predict(X_instance)[0]
true_value = y_test.iloc[index] if isinstance(y_test, pd.Series) else y_test[index]

print("\nPredicted value: " + str(pred_value))
print("Actual value: " + str(true_value))

##Local EBM Explainability

In [ ]:
ebm_local = ebm.explain_local(X_test, y_test)
show(ebm_local)

##SHAP Waterfalls

In [ ]:
shap.initjs()
shap.force_plot(dt_explainer.expected_value, dt_shap_values[index, :], X_test_df.iloc[index])

In [ ]:
shap.initjs()
shap.force_plot(ebm_explainer.expected_value, ebm_shap_values[index, :], X_test_df.iloc[index])

In [ ]:
shap.initjs()
shap.force_plot(cat_explainer.expected_value, cat_shap_values[index, :], X_test_df.iloc[index])

In [ ]:
shap.initjs()
shap.force_plot(lgbm_explainer.expected_value, lgbm_shap_values[index, :], X_test_df.iloc[index])

#SHAP Evaluation

##Setting up Importances for EBM

In [ ]:
term_names = np.array(ebm.term_names_)
term_importances = np.array(ebm.term_importances())
print(term_names)

In [ ]:
main_mask = np.array([' & ' not in name for name in term_names])
main_features = term_names[main_mask]
main_importances = term_importances[main_mask]

In [ ]:
ebm_importances = np.zeros(len(X.columns))
for i, feature in enumerate(X.columns):
    if feature in main_features:
        idx = list(main_features).index(feature)
        ebm_importances[i] = main_importances[idx]

In [ ]:
models_dict = {
    'DT': (dt, dt_explainer, dt_shap_values, dt.feature_importances_, dt_sage_values.values),
    'EBM': (ebm, ebm_explainer, ebm_shap_values, ebm_importances, ebm_sage_values.values),
    'CAT': (cat, cat_explainer, cat_shap_values, cat.feature_importances_, cat_sage_values.values),
    'LGBM': (lgbm, lgbm_explainer, lgbm_shap_values, lgbm.feature_importances_, lgbm_sage_values.values)
}

##Fidelity (Correlation between model and SHAP feature importances)

In [ ]:
def fidelity(model_importance, shap_values):
    shap_importance = np.abs(shap_values).mean(axis=0)
    return spearmanr(model_importance, shap_importance)[0]

##Consistency (Entropy of top feature across instances)

In [ ]:
def shap_consistency(shap_values, feature_names):
    top_feature = np.argmax(np.abs(shap_values), axis=1)

    value, counts = np.unique(top_feature, return_counts=True)
    probs = counts / len(top_feature)
    entropy = -np.sum(probs * np.log(probs))

    dominant_feature = feature_names[value[np.argmax(counts)]]
    dominant_percent = counts.max() / len(top_feature)

    return entropy, dominant_feature, dominant_percent

##Concentration (Entropy of explanation values)

In [ ]:
def shap_concentration(shap_values, feature_names):
    vals = np.abs(shap_values).mean(axis=0)

    total = vals.sum()
    probs = vals / total
    entropy = -np.sum(probs * np.log(probs + 1e-12))

    dominant_idx = np.argmax(vals)
    dominant_feature = feature_names[dominant_idx]
    dominant_percent = vals[dominant_idx] / total

    return entropy, dominant_feature, dominant_percent

In [ ]:
def sage_concentration(sage_values, feature_names):
    vals = np.abs(sage_values)

    total = vals.sum()
    probs = vals / total
    entropy = -np.sum(probs * np.log(probs + 1e-12))

    dominant_idx = np.argmax(vals)
    dominant_feature = feature_names[dominant_idx]
    dominant_percent = vals[dominant_idx] / total

    return entropy, dominant_feature, dominant_percent

##Robustness (Test if SHAP values remain stable under small perturbations)

In [ ]:
def local_robustness(explainer, X_sample_df, n_perturbations=10, noise_std=0.1, seed=100):
    rng = np.random.RandomState(seed)
    stabilities = []

    for i in range(len(X_sample_df)):
        instance_df = X_sample_df.iloc[i:i+1]
        base_shap = explainer.shap_values(instance_df)

        corrs = []
        for _ in range(n_perturbations):
            noise = rng.normal(0, noise_std, instance_df.shape)
            perturbed_df = pd.DataFrame(instance_df.values + noise, columns=instance_df.columns)
            perturbed_shap = explainer.shap_values(perturbed_df)
            corr = spearmanr(np.abs(base_shap).ravel(), np.abs(perturbed_shap).ravel())[0]
            corrs.append(corr)

        stabilities.append(np.mean(corrs))

    return np.mean(stabilities)

In [ ]:
def global_robustness(model, background, X_test_df, n_perturbations=10, noise_std=0.1, seed=100):
    rng = np.random.RandomState(seed)

    if model in [dt, cat, lgbm]:
        explainer = shap.TreeExplainer(model)
    elif model == ebm:
        explainer = shap.KernelExplainer(ebm_predict_wrapper, background, seed=seed)

    base_shap = explainer.shap_values(X_test_df)
    base_global = np.abs(base_shap).mean(axis=0)

    correlations = []

    for _ in range(n_perturbations):
        noise = rng.normal(0, noise_std, X_test_df.shape)
        X_test_pert = pd.DataFrame(X_test_df.values + noise, columns=X_test_df.columns)
        pert_shap = explainer.shap_values(X_test_pert)

        pert_global = np.mean(np.abs(pert_shap), axis=0)
        correlations.append(spearmanr(base_global, pert_global)[0])

    return float(np.mean(correlations))

In [ ]:
def sage_robustness(model, X_train, X_test, y_test, n_perturbations=10, noise_std=0.1, seed=100, state=100, max_background=256):
    rng = np.random.RandomState(seed)

    n_bg = min(max_background, len(X_train))
    idx = rng.choice(len(X_train), size=n_bg, replace=False)

    X_bg = X_train.iloc[idx].values

    imputer = sage.MarginalImputer(model, X_bg)
    estimator = sage.PermutationEstimator(imputer, loss='mse', random_state=state)
    base_vals = estimator(X_test.values, y_test.values).values

    correlations = []

    for _ in range(n_perturbations):
        noise = rng.normal(0, noise_std, X_test.shape)
        X_test_pert = X_test.values + noise

        pert_vals = estimator(X_test_pert, y_test.values).values

        corr = spearmanr(np.abs(base_vals), np.abs(pert_vals))[0]
        correlations.append(1.0 if np.isnan(corr) else corr)

    return float(np.mean(correlations))

##Sufficiency (Test if top-k SHAP features preserve predictions)




In [ ]:
def sufficiency(model, X_test, shap_values, feature_names, k=5, n_samples=30):
    abs_errors = []
    rel_errors = []
    all_top_features = []

    for i in range(min(n_samples, len(X_test))):
        X_instance = X_test.iloc[i:i+1]
        original_pred = model.predict(X_instance)[0]

        shap_vals = shap_values[i, :]

        top_k_indices = np.argsort(np.abs(shap_vals))[-k:]
        all_top_features.extend(top_k_indices)

        masked_instance = pd.DataFrame(np.zeros((1, X_test.shape[1])), columns=X_test.columns)
        masked_instance.iloc[0, top_k_indices] = X_test.iloc[i, top_k_indices]

        masked_pred = model.predict(masked_instance)[0]

        abs_error = abs(original_pred - masked_pred)
        rel_error = abs_error / (abs(original_pred) + 1e-10)

        abs_errors.append(abs_error)
        rel_errors.append(rel_error)

    feature_counter = Counter(all_top_features)

    feature_usage = []
    for feat_idx in range(len(feature_names)):
        count = feature_counter.get(feat_idx, 0)
        percent = (count / n_samples) * 100
        feature_usage.append({'feature': feature_names[feat_idx], 'count': count, 'percentage': percent})

    feature_usage_df = pd.DataFrame(feature_usage).sort_values('count', ascending=False)

    top_features_dict = {feature_names[idx]: count for idx, count in feature_counter.most_common(k)}

    threshold = 0.10
    predictions_maintained = np.mean([err < threshold for err in rel_errors])

    return {
        'avg_abs_error': np.mean(abs_errors),
        'avg_rel_error': np.mean(rel_errors),
        'percent_maintained': predictions_maintained,
        'top_features_used': top_features_dict,
        'feature_usage_df': feature_usage_df
    }

##Completeness

In [ ]:
def completeness(model, X_test, shap_values, feature_names, k=5, n_samples=30):
    abs_errors = []
    rel_errors = []
    all_removed_features = []

    for i in range(min(n_samples, len(X_test))):
        X_instance = X_test.iloc[i:i+1]
        original_pred = model.predict(X_instance)[0]

        shap_vals = shap_values[i, :]
        top_k_indices = np.argsort(np.abs(shap_vals))[-k:]
        all_removed_features.extend(top_k_indices)

        masked_instance = X_instance.copy()
        masked_instance.iloc[0, top_k_indices] = 0

        masked_pred = model.predict(masked_instance)[0]

        abs_error = abs(original_pred - masked_pred)
        rel_error = abs_error / (abs(original_pred) + 1e-10)

        abs_errors.append(abs_error)
        rel_errors.append(rel_error)

    feature_counter = Counter(all_removed_features)

    feature_usage = []
    for feat_idx in range(len(feature_names)):
        count = feature_counter.get(feat_idx, 0)
        percent = (count / n_samples) * 100
        feature_usage.append({'feature': feature_names[feat_idx], 'count': count, 'percentage': percent})

    feature_usage_df = pd.DataFrame(feature_usage).sort_values('count', ascending=False)

    top_features_dict = {feature_names[idx]: count for idx, count in feature_counter.most_common(k)}

    threshold = 0.10
    predictions_maintained = np.mean([err < threshold for err in rel_errors])

    return {
        'avg_abs_error': np.mean(abs_errors),
        'avg_rel_error': np.mean(rel_errors),
        'percent_maintained': predictions_maintained,
        'top_features_removed': top_features_dict,
        'feature_usage_df': feature_usage_df
    }

##SHAP Evaluation Results

In [ ]:
results = {}

for model_name, (model, explainer, shap_vals, feat_imp, sage_vals) in models_dict.items():
    fl_score = fidelity(feat_imp, shap_vals)

    sage_fl_score = spearmanr(feat_imp, sage_vals)[0]

    shap_sage_agree = fidelity(sage_vals, shap_vals)

    shap_consistency_entropy, shap_consistency_dominant_feature, shap_consistency_dominant_percent = shap_consistency(shap_vals, X.columns)
    shap_consistency_score = 1 - (shap_consistency_entropy / np.log(len(X.columns)))#min-max normalization

    shap_concentration_entropy, shap_concentration_dominant_feature, shap_concentration_dominant_percent = shap_concentration(shap_vals, X.columns)
    shap_concentration_score = 1 - (shap_concentration_entropy / np.log(len(X.columns)))

    sage_concentration_entropy, sage_concentration_dominant_feature, sage_concentration_dominant_percent = sage_concentration(sage_vals, X.columns)
    sage_concentration_score = 1 - (sage_concentration_entropy / np.log(len(X.columns)))

    X_local = X_test_df.iloc[:20]
    local_rb_baseline = local_robustness(explainer, X_local, seed=SEED)
    local_rb_more_perturbations = local_robustness(explainer, X_local, n_perturbations=30, seed=SEED)
    local_rb_higher_noise = local_robustness(explainer, X_local, noise_std=0.3, seed=SEED)

    X_global = X_test_df.iloc[:256]
    background = shap.sample(X_train, 100, random_state=SEED)
    global_rb_baseline = global_robustness(model, background, X_global, seed=SEED)
    global_rb_more_perturbations = global_robustness(model, background, X_global, n_perturbations=30, seed=SEED)
    global_rb_higher_noise = global_robustness(model, background, X_global, noise_std=0.3, seed=SEED)

    X_test_sub = X_test.iloc[:256]
    y_test_sub = y_test.iloc[:256]
    sage_rb_baseline = sage_robustness(model, X_train, X_test_sub, y_test_sub, seed=SEED)
    sage_rb_perturbations = sage_robustness(model, X_train, X_test_sub, y_test_sub, n_perturbations=30, seed=SEED)
    sage_rb_noise = sage_robustness(model, X_train, X_test_sub, y_test_sub,noise_std=0.3, seed=SEED)

    sufficiency_k1 = sufficiency(model, X_test, shap_vals, X.columns, k=1)
    sufficiency_k3 = sufficiency(model, X_test, shap_vals, X.columns, k=3)
    sufficiency_k5 = sufficiency(model, X_test, shap_vals, X.columns, k=5)
    sufficiency_k8 = sufficiency(model, X_test, shap_vals, X.columns, k=8)

    completeness_k1 = completeness(model, X_test, shap_vals, X.columns, k=1)
    completeness_k3 = completeness(model, X_test, shap_vals, X.columns, k=3)
    completeness_k5 = completeness(model, X_test, shap_vals, X.columns, k=5)
    completeness_k8 = completeness(model, X_test, shap_vals, X.columns, k=8)

    results[model_name] = {
        'fidelity': fl_score,
        'sage_fidelity': sage_fl_score,
        'shap_sage_agreement': shap_sage_agree,
        'shap_consistency': shap_consistency_score,
        'shap_consistency_dominant_feature': shap_consistency_dominant_feature,
        'shap_consistency_dominant_percent': shap_consistency_dominant_percent,
        'shap_concentration': shap_concentration_score,
        'shap_concentration_dominant_feature': shap_concentration_dominant_feature,
        'shap_concentration_dominant_percent': shap_concentration_dominant_percent,
        'sage_concentration': sage_concentration_score,
        'sage_concentration_dominant_feature': sage_concentration_dominant_feature,
        'sage_concentration_dominant_percent': sage_concentration_dominant_percent,
        'local_robustness_baseline': local_rb_baseline,
        'local_robustness_more_perturbations': local_rb_more_perturbations,
        'local_robustness_higher_noise': local_rb_higher_noise,
        'global_robustness_baseline': global_rb_baseline,
        'global_robustness_more_perturbations': global_rb_more_perturbations,
        'global_robustness_higher_noise': global_rb_higher_noise,
        'sage_robustness_baseline': sage_rb_baseline,
        'sage_robustness_more_perturbations': sage_rb_perturbations,
        'sage_robustness_higher_noise': sage_rb_noise,
        'sufficiency_k1': sufficiency_k1,
        'sufficiency_k3': sufficiency_k3,
        'sufficiency_k5': sufficiency_k5,
        'sufficiency_k8': sufficiency_k8,
        'completeness_k1': completeness_k1,
        'completeness_k3': completeness_k3,
        'completeness_k5': completeness_k5,
        'completeness_k8': completeness_k8
    }

##Results per Model

In [ ]:
model_names = ['DT', 'EBM', 'CAT', 'LGBM']

##Fidelity Plots

In [ ]:
fl_scores = [results[m]['fidelity'] for m in model_names]
plt.bar(model_names, fl_scores)
plt.ylabel('Correlation')
plt.title('SHAP Fidelity')
plt.show()

In [ ]:
sage_fl_scores = [results[m]['sage_fidelity'] for m in model_names]
plt.bar(model_names, sage_fl_scores)
plt.ylabel('Correlation')
plt.title('Sage Fidelity')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['fidelity', 'sage_fidelity']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##SHAP-SAGE Agreement Plot

In [ ]:
agreement_scores = [results[m]['shap_sage_agreement'] for m in model_names]
plt.bar(model_names, agreement_scores)
plt.ylabel('Correlation')
plt.title('SHAP-SAGE Agreement')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    print("shap_sage_agreement: " + str(results[model]['shap_sage_agreement']))
    print("")

##Consistency Plots

In [ ]:
shap_consistency_scores = [results[m]['shap_consistency'] for m in model_names]
plt.bar(model_names, shap_consistency_scores)
plt.ylabel('Correlation')
plt.title('SHAP Consistency Scores')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['shap_consistency_dominant_feature', 'shap_consistency_dominant_percent']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Concentration Plots

In [ ]:
shap_concentration_scores = [results[m]['shap_concentration'] for m in model_names]
plt.bar(model_names, shap_concentration_scores)
plt.ylabel('Correlation')
plt.title('Shap Concentration Scores')
plt.show()

In [ ]:
sage_concentration_scores = [results[m]['sage_concentration'] for m in model_names]
plt.bar(model_names, sage_concentration_scores)
plt.ylabel('Correlation')
plt.title('Sage Concentration Scores')
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['shap_concentration_dominant_feature', 'shap_concentration_dominant_percent',
                   'sage_concentration_dominant_feature', 'sage_concentration_dominant_percent']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

##Robustness Testing Results for each Model

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, [results[m]['local_robustness_baseline'] for m in model_names], width, label='RB 1')
rects2 = ax.bar(x - 0.5*width, [results[m]['local_robustness_more_perturbations'] for m in model_names], width, label='RB 2')
rects3 = ax.bar(x + 0.5*width, [results[m]['local_robustness_higher_noise'] for m in model_names], width, label='RB 3')

ax.set_ylabel('Correlation')
ax.set_title('Local SHAP Robustness by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(['RB 1 (n_perturbations=10, noise_std=0.1)', 'RB 2 (n_perturbations=30, noise_std=0.1)', 'RB 3 (n_perturbations=10, noise_std=0.3)'])

plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['local_robustness_baseline', 'local_robustness_more_perturbations', 'local_robustness_higher_noise']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, [results[m]['global_robustness_baseline'] for m in model_names], width, label='RB 1')
rects2 = ax.bar(x - 0.5*width, [results[m]['global_robustness_more_perturbations'] for m in model_names], width, label='RB 2')
rects3 = ax.bar(x + 0.5*width, [results[m]['global_robustness_higher_noise'] for m in model_names], width, label='RB 3')

ax.set_ylabel('Correlation')
ax.set_title('Global SHAP Robustness by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(['RB 1 (n_perturbations=10, noise_std=0.1)', 'RB 2 (n_perturbations=30, noise_std=0.1)', 'RB 3 (n_perturbations=10, noise_std=0.3)'])

plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['global_robustness_baseline', 'global_robustness_more_perturbations', 'global_robustness_higher_noise']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, [results[m]['sage_robustness_baseline'] for m in model_names], width, label='RB 1')
rects2 = ax.bar(x - 0.5*width, [results[m]['sage_robustness_more_perturbations'] for m in model_names], width, label='RB 2')
rects3 = ax.bar(x + 0.5*width, [results[m]['sage_robustness_higher_noise'] for m in model_names], width, label='RB 3')

ax.set_ylabel('Correlation')
ax.set_title('Sage Robustness by Model')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(['RB 1 (n_perturbations=10, noise_std=0.1)', 'RB 2 (n_perturbations=30, noise_std=0.1)', 'RB 3 (n_perturbations=10, noise_std=0.3)'])

plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model))
    for metric in ['sage_robustness_baseline', 'sage_robustness_more_perturbations', 'sage_robustness_higher_noise']:
        print(str(metric) + ": " + str(results[model][metric]))
    print("")

Decision Tree Sage Robustness:
0.4825593395252838,
0.5472308221534228,
0.6045407636738905

EBM Sage Robustness:
0.6579979360165119,
0.6205022359821123,
0.6476780185758513

CatBoost Sage Robustness:
0.39360165118679047,
0.38341933264533884,
0.4008255933952528

LightGBM Sage Robustness:
0.6425180598555212,
0.6452012383900929,
0.6117647058823529

##Suffiency Testing Results for each Model

In [ ]:
dt_top5 = set(results['DT']['sufficiency_k5']['feature_usage_df'].head(5)['feature'])
ebm_top5 = set(results['EBM']['sufficiency_k5']['feature_usage_df'].head(5)['feature'])
cat_top5 = set(results['CAT']['sufficiency_k5']['feature_usage_df'].head(5)['feature'])
lgbm_top5 = set(results['LGBM']['sufficiency_k5']['feature_usage_df'].head(5)['feature'])

print("Decision Tree Top 5 Features: " + str(dt_top5))
print("EBM Top 5 Features: " + str(ebm_top5))
print("CatBoost Top 5 Features: " + str(cat_top5))
print("LightGBM Top 5 Features: " + str(lgbm_top5))

In [ ]:
percent_maintained_k1 = [results[m]['sufficiency_k1']['percent_maintained'] for m in model_names]
percent_maintained_k3 = [results[m]['sufficiency_k3']['percent_maintained'] for m in model_names]
percent_maintained_k5 = [results[m]['sufficiency_k5']['percent_maintained'] for m in model_names]
percent_maintained_k8 = [results[m]['sufficiency_k8']['percent_maintained'] for m in model_names]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, percent_maintained_k1, width, label='k=1')
rects2 = ax.bar(x - 0.5*width, percent_maintained_k3, width, label='k=3')
rects3 = ax.bar(x + 0.5*width, percent_maintained_k5, width, label='k=5')
rects4 = ax.bar(x + 1.5*width, percent_maintained_k8, width, label='k=8')

ax.set_ylabel('% Predictions Maintained (<10% error)')
ax.set_title('Sufficiency Test: Prediction Preservation for Different k')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()

plt.tight_layout()
plt.show()

##Completeness Testing Results for each Model

In [ ]:
percent_maintained_k1 = [results[m]['completeness_k1']['percent_maintained'] for m in model_names]
percent_maintained_k3 = [results[m]['completeness_k3']['percent_maintained'] for m in model_names]
percent_maintained_k5 = [results[m]['completeness_k5']['percent_maintained'] for m in model_names]
percent_maintained_k8 = [results[m]['completeness_k8']['percent_maintained'] for m in model_names]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.2

rects1 = ax.bar(x - 1.5*width, percent_maintained_k1, width, label='k=1')
rects2 = ax.bar(x - 0.5*width, percent_maintained_k3, width, label='k=3')
rects3 = ax.bar(x + 0.5*width, percent_maintained_k5, width, label='k=5')
rects4 = ax.bar(x + 1.5*width, percent_maintained_k8, width, label='k=8')

ax.set_ylabel('% Predictions Maintained (<10% error)')
ax.set_title('Completeness Test: Prediction Preservation for Different k')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()

plt.tight_layout()
plt.show()

##R2 Score vs Explainability Plot (Through mean values of metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

explain_scores = {
    m: np.mean([
        results[m]['fidelity'],
        results[m]['shap_consistency'],
        results[m]['shap_concentration'],
        results[m]['local_robustness_baseline'],
        results[m]['global_robustness_baseline'],
        results[m]['sufficiency_k5']['percent_maintained'],
        results[m]['completeness_k3']['percent_maintained']
    ]) for m in model_names
}

r2_scores = {
    'DT': r2_score(y_test, dt_pred),
    'EBM': r2_score(y_test, ebm_pred),
    'CAT': r2_score(y_test, cat_pred),
    'LGBM': r2_score(y_test, lgbm_pred)
}

for model in model_names:
    ax.scatter(explain_scores[model], r2_scores[model])
    ax.annotate(model, (explain_scores[model], r2_scores[model]))

ax.set_xlabel('Explainability Score (0-1)')
ax.set_ylabel('Model R2 Score')
ax.set_title('R2 Score vs Explainability Tradeoff')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for model in model_names:
    print("Model: " + str(model) + ", R2 Score: " + str(r2_scores[model]) + ", Explainability Score:" + str(explain_scores[model]))